# Gold: Menggabungkan Semua Bronze → DuckLake (Star Schema)

Notebook ini **menggabungkan** 4 database bronze menjadi **satu DuckLake** (lapisan gold): pembersihan data,
pencocokan nama HP antar sumber, penyamaan jumlah data Tokopedia vs Shopee, dan penyusunan star schema — semua
logikanya sama dengan yang sudah diuji di notebook gabungan sebelumnya, hanya sumbernya sekarang dari `ATTACH`
beberapa file `.duckdb`, bukan CSV.

```
data/gsmarena.duckdb  ─┐
data/tokopedia.duckdb ─┼─ ATTACH (read-only) ──► bersihkan, cocokkan, seimbangkan ──► CREATE TABLE lake.*
data/shopee.duckdb    ─┤                                  (notebook ini)              (star schema)
data/youtube.duckdb   ─┘ (opsional, lihat catatan)
                                                                     │
                                                                     ▼
                                                    data/lakehouse/lakehouse.ducklake  ──► Streamlit
```

## Yang perlu Anda siapkan dulu
1. **`data/gsmarena.duckdb` dan `data/tokopedia.duckdb` harus sudah berisi data** (wajib).
2. **`data/shopee.duckdb`** sebaiknya sudah ada juga (untuk perbandingan harga); jika belum, notebook tetap
   jalan tapi perbandingan Tokopedia-vs-Shopee otomatis dilewati dan diberi tahu di output.
3. **`data/youtube.duckdb`** bersifat **opsional** — notebook sumber YouTube belum disesuaikan ke arsitektur
   DuckDB ini, jadi bagian sentimen di sini **dilewati otomatis** jika filenya belum ada / skemanya belum cocok.
   Akan diintegrasikan begitu notebook YouTube-nya dibangun ulang.
4. **Koneksi internet** dibutuhkan **sekali saja** saat pertama kali `INSTALL ducklake` (mengunduh ekstensi DuckDB,
   lalu tersimpan di cache lokal dan bisa dipakai offline setelahnya).

> ⚠️ Bagian `INSTALL ducklake` / `ATTACH 'ducklake:...'` di notebook ini **belum bisa saya coba jalankan secara
> langsung** (lingkungan saya tidak punya akses ke `extensions.duckdb.org`), jadi bagian itu saya susun mengikuti
> dokumentasi resmi DuckLake, bukan hasil uji coba langsung. Seluruh logika pembersihan, pencocokan nama HP,
> penyamaan data, dan pembentukan star schema **sudah saya uji penuh** dengan database bronze tiruan.

## 1. Instalasi & Import Library

In [ ]:
%pip install -q duckdb pandas

In [2]:
import re
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. Konfigurasi

In [ ]:
DB_DIR = Path("database")
GSMARENA_DB = DB_DIR / "gsmarena.duckdb"
TOKOPEDIA_DB = DB_DIR / "tokopedia.duckdb"
SHOPEE_DB = DB_DIR / "shopee.duckdb"
YOUTUBE_DB = DB_DIR / "youtube.duckdb"   # opsional

LAKE_DIR = Path("lakehouse")
LAKE_DIR.mkdir(parents=True, exist_ok=True)
LAKE_CATALOG = LAKE_DIR / "lakehouse.ducklake"     # metadata DuckLake
LAKE_DATA_PATH = LAKE_DIR / "files"                # folder file Parquet milik DuckLake

GITHUB_WARN_MB = 50

# ---------------- Aturan bisnis lapisan gold (sebelumnya sudah diuji di notebook gabungan) ----------------
MIN_YEAR = 2025                 # filter ketat sekali lagi (bronze GSMArena bisa menyimpan sedikit HP di bawah ini)
N_PHONES_COMPARE = 60           # jumlah HP yang dibandingkan harganya (SAMA di Tokopedia & Shopee)
LISTINGS_PER_PHONE = 10         # K: jumlah listing per HP per platform (SAMA untuk kedua platform)
MIN_LISTINGS_PER_PHONE = 5
MIN_PHONE_PRICE = 300_000
SECOND_HAND_WORDS = ["second", "seken", "bekas", "preloved", "refurbish", "rekondisi", "no minus"]

VARIANT_TOKENS = {"pro", "plus", "ultra", "max", "lite", "fe", "mini", "se", "edge", "neo", "prime",
                  "turbo", "play", "note", "fold", "flip"}

## 3. ATTACH Semua Database Bronze (Read-Only)

In [4]:
con = duckdb.connect(":memory:")   # gold dibangun di memori dulu, baru ditulis ke DuckLake di bagian 8

for name, path, wajib in [("gsm", GSMARENA_DB, True), ("tokopedia", TOKOPEDIA_DB, True),
                          ("shopee", SHOPEE_DB, False), ("yt", YOUTUBE_DB, False)]:
    if path.exists():
        con.execute(f"ATTACH '{path}' AS {name} (READ_ONLY)")
        print(f"✔ {name} terhubung ({path})")
    elif wajib:
        raise RuntimeError(f"{path} tidak ditemukan — jalankan notebook scraping-nya terlebih dahulu.")
    else:
        print(f"– {name} dilewati ({path} belum ada)")

HAS_SHOPEE = SHOPEE_DB.exists()
HAS_YOUTUBE = YOUTUBE_DB.exists()

✔ gsm terhubung (data\gsmarena.duckdb)
✔ tokopedia terhubung (data\tokopedia.duckdb)
✔ shopee terhubung (data\shopee.duckdb)
– yt dilewati (data\youtube.duckdb belum ada)


In [5]:
n_specs = con.execute("SELECT count(*) FROM gsm.phone_specs").fetchone()[0]
n_tokped = con.execute("SELECT count(*) FROM tokopedia.listings").fetchone()[0]
n_shopee = con.execute("SELECT count(*) FROM shopee.listings").fetchone()[0] if HAS_SHOPEE else 0
print(f"phone_specs (GSMArena): {n_specs}\nlistings (Tokopedia):   {n_tokped}\nlistings (Shopee):      {n_shopee}")

phone_specs (GSMArena): 282
listings (Tokopedia):   1451
listings (Shopee):      606


## 4. Bersihkan Spesifikasi HP (GSMArena)
Sama seperti sebelumnya: memisahkan chipset, mem-parsing RAM/storage/baterai ke angka, membuang HP batal
rilis/rumor, dan memastikan jendela tahun (`MIN_YEAR`) benar-benar ketat.

In [6]:
def is_missing(text):
    # NULL dari DuckDB bisa muncul sebagai None ATAU nan (float) di pandas — bool(nan) justru True,
    # jadi "if text" / "if not text" saja TIDAK CUKUP untuk menangkap nilai kosong.
    return text is None or (isinstance(text, float) and pd.isna(text))


def clean_chipset(text):
    if is_missing(text):
        return "Unknown"
    return re.sub(r"\(.*?\)", "", text).strip()


def chipset_brand(text):
    for key in ["Snapdragon", "Exynos", "Dimensity", "Helio", "Tensor", "Unisoc", "Kirin", "Apple",
                "Bionic", "MediaTek", "Spreadtrum", "Qualcomm"]:
        if key.lower() in (text or "").lower():
            return "Apple" if key == "Bionic" else key
    return "Other"


def parse_ram_gb(text):
    if is_missing(text):
        return None
    vals = [float(x) for x in re.findall(r"(\d+)\s*GB\s*RAM", text, flags=re.I)]
    vals += [int(x) / 1024 for x in re.findall(r"(\d+)\s*MB\s*RAM", text, flags=re.I)]
    return round(max(vals), 3) if vals else None


def parse_storage_gb(text):
    if is_missing(text):
        return None
    vals = []
    for num, unit in re.findall(r"(\d+)\s*(TB|GB|MB)\b(?!\s*RAM)", text, flags=re.I):
        n, u = int(num), unit.upper()
        vals.append(n * 1024 if u == "TB" else n if u == "GB" else n / 1024)
    return round(max(vals), 3) if vals else None


def parse_battery_mah(text):
    if is_missing(text):
        return None
    m = re.search(r"(\d{3,5})\s*mAh", text, flags=re.I)
    return int(m.group(1)) if m else None


def normalize_name(text):
    if is_missing(text):
        return ""
    t = text.lower().replace("+", " plus ")
    t = re.sub(r"\b\d+\s*/\s*\d+\s*(gb)?\b", " ", t)
    t = re.sub(r"\b\d+\s*(gb|tb|mah)\b", " ", t)
    t = re.sub(r"[^a-z0-9 ]", " ", t)
    noise = {"smartphone", "hp", "handphone", "garansi", "resmi", "ram", "rom", "new", "original", "ori", "baru", "bnib"}
    return " ".join(w for w in t.split() if w not in noise)


specs_raw = con.execute("SELECT * FROM gsm.phone_specs").df()
specs_raw = specs_raw[~specs_raw["released_raw"].fillna("").str.contains(r"cancel|rumor|exp\. release", case=False, regex=True)]
specs_raw = specs_raw[specs_raw["release_year"].notna() & (specs_raw["release_year"] >= MIN_YEAR)]

df_specs = pd.DataFrame({
    "brand": specs_raw["brand"], "phone_name": specs_raw["phone_name"].str.strip(),
    "chipset": specs_raw["chipset_raw"].apply(clean_chipset),
    "ram_gb": specs_raw["internalmemory_raw"].apply(parse_ram_gb),
    "storage_gb": specs_raw["internalmemory_raw"].apply(parse_storage_gb),
    "battery_mah": specs_raw["battery_raw"].apply(parse_battery_mah),
    "release_year": specs_raw["release_year"].astype(int),
    "gsmarena_url": specs_raw["gsmarena_url"],
})
df_specs["chipset_brand"] = df_specs["chipset"].apply(chipset_brand)
df_specs = df_specs.drop_duplicates(subset="phone_name").reset_index(drop=True)

print(f"{len(df_specs)} HP siap dipakai (rilis >= {MIN_YEAR})")
df_specs.head()

270 HP siap dipakai (rilis >= 2025)


,brand,phone_name,chipset,ram_gb,storage_gb,battery_mah,release_year,gsmarena_url,chipset_brand
0,Apple,Apple iPhone Duo,Apple A20 Pro,12.0,2048,NaN,2026,https://www.gsmarena.com/apple_iphone_duo_fold...,Apple
1,Apple,Apple iPhone 18 Pro Max,Apple A20 Pro,12.0,2048,5391.0,2026,https://www.gsmarena.com/apple_iphone_18_pro_m...,Apple
2,Apple,Apple iPhone 18 Pro,Apple A20 Pro,12.0,2048,4056.0,2026,https://www.gsmarena.com/apple_iphone_18_pro-1...,Apple
3,Apple,Apple iPhone 17e,Apple A19,8.0,512,4005.0,2026,https://www.gsmarena.com/apple_iphone_17e-1448...,Apple
4,Apple,Apple iPhone 17 Pro Max,Apple A19 Pro,12.0,2048,4823.0,2025,https://www.gsmarena.com/apple_iphone_17_pro_m...,Apple


## 5. Bersihkan Data Penjualan (Tokopedia + Shopee)
Mengubah harga/rating/terjual dari teks ke angka, membuang HP bekas (aksesoris sudah disaring sejak bronze),
dan menghapus listing dobel.

In [7]:
def clean_price(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    m = re.search(r"\d[\d\.,]*", str(text))
    if not m:
        return None
    digits = re.sub(r"[^\d]", "", m.group(0))
    return int(digits) if digits else None


def parse_sold(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    t = str(text).lower().replace("terjual", "").replace("/bulan", "").replace("+", "").strip()
    m = re.search(r"([\d\.,]+)\s*(rb|jt)?", t)
    if not m:
        return None
    raw, suffix = m.group(1), m.group(2)
    try:
        if suffix:
            val = float(raw.replace(".", "").replace(",", ".")) if "," in raw else (
                  float(raw) if re.fullmatch(r"\d+\.\d{1,2}", raw) else float(raw.replace(".", "")))
            num = val * (1000 if suffix == "rb" else 1_000_000)
        else:
            num = float(raw.replace(".", "").replace(",", ""))
    except ValueError:
        return None
    return int(num)


def parse_rating(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    m = re.search(r"\d+(?:[.,]\d+)?", str(text))
    return float(m.group(0).replace(",", ".")) if m else None


def parse_discount_pct(text):
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    m = re.search(r"(\d{1,2})\s*%", str(text))
    return int(m.group(1)) if m else None


def clean_sales(df, platform):
    out = df.rename(columns={"keyword": "search_phone", "scraped_at": "scrape_date"}).copy()
    out["platform"] = platform
    out["product_url"] = out["product_url"].apply(lambda u: u.split("?")[0] if isinstance(u, str) else None)
    out = out.dropna(subset=["product_name"])

    low = out["product_name"].str.lower()
    out = out[~low.apply(lambda s: any(w in s for w in SECOND_HAND_WORDS))]

    out["price_discount"] = out["price_discount_raw"].apply(clean_price)
    out["price_original"] = pd.to_numeric(out["price_original_raw"].apply(clean_price), errors="coerce")
    pct = out["discount_label_raw"].apply(parse_discount_pct)
    est = out["price_original"].isna() & pct.notna() & (pct > 0) & out["price_discount"].notna()
    out["is_original_estimated"] = est
    if est.any():
        out.loc[est, "price_original"] = (out.loc[est, "price_discount"] / (1 - pct[est] / 100)).round()

    out = out[out["price_discount"].notna() & (out["price_discount"] >= MIN_PHONE_PRICE)].copy()
    out["price_original"] = out[["price_original", "price_discount"]].max(axis=1)
    out["price_discount"] = out["price_discount"].astype(int)
    out["price_original"] = out["price_original"].round().astype(int)
    out["discount_pct"] = ((1 - out["price_discount"] / out["price_original"]) * 100).round(1)

    out["rating"] = out["rating_raw"].apply(parse_rating)
    out["units_sold"] = out["sold_raw"].apply(parse_sold).fillna(0).astype(int)
    out["seller_name"] = out["seller_name"].fillna("Unknown")
    out["seller_location"] = out["seller_location"].fillna("Unknown")

    key = out["product_url"].fillna(out["platform"] + "|" + out["product_name"] + "|" + out["seller_name"])
    out = out.loc[~key.duplicated()].copy()
    out["scrape_date"] = pd.to_datetime(out["scrape_date"]).dt.date

    return out.reset_index(drop=True)[[
        "platform", "search_phone", "product_name", "product_url", "price_original", "price_discount",
        "discount_pct", "is_original_estimated", "rating", "units_sold", "seller_name", "seller_location", "scrape_date"]]


tk_raw = con.execute("SELECT * FROM tokopedia.listings").df()
df_sales_parts = [clean_sales(tk_raw, "Tokopedia")]
if HAS_SHOPEE:
    sp_raw = con.execute("SELECT * FROM shopee.listings").df()
    df_sales_parts.append(clean_sales(sp_raw, "Shopee"))

df_sales_all = pd.concat(df_sales_parts, ignore_index=True)
print(df_sales_all["platform"].value_counts().to_string())
df_sales_all.head()

platform
Tokopedia    951
Shopee       307


,platform,search_phone,product_name,product_url,price_original,price_discount,discount_pct,is_original_estimated,rating,units_sold,seller_name,seller_location,scrape_date
0,Tokopedia,Apple iPhone Duo,15% Apple iPhone 18 Pro / 18 Pro Max 2TB 1TB 5...,https://www.tokopedia.com/gofone/apple-iphone-...,43370000,36784000,15.2,False,NaN,0,gofone,Unknown,2026-10-01
1,Tokopedia,Apple iPhone Duo,33% PreOrder OPEN PO iPhone DUO 256GB 512GB 1T...,https://www.tokopedia.com/wowgadget/terjual-64...,93500000,62499000,33.2,False,4.9,60,wowgadget,Unknown,2026-10-01
2,Tokopedia,Apple iPhone Duo,16% [ DGI Gadget ] Apple iPhone 17 256GB Garan...,https://www.tokopedia.com/dgi-gadget/dgi-gadge...,22499000,18990000,15.6,False,4.9,100,dgi-gadget,Unknown,2026-10-01
3,Tokopedia,Apple iPhone Duo,"Apple iPhone Duo 7.6"" inch Super Retina XDR fo...",https://www.tokopedia.com/digitalauthorized/ap...,65999000,65999000,0.0,False,NaN,1,digitalauthorized,Unknown,2026-10-01
4,Tokopedia,Apple iPhone Duo,36% PreOrder OPEN PO iPhone DUO 256GB 512GB 1T...,https://www.tokopedia.com/techstory/apple-ipho...,99360000,63349000,36.2,False,5.0,100,techstory,Unknown,2026-10-01


## 6. Cocokkan Nama HP & Samakan Jumlah Data Antar Platform
Sama seperti sebelumnya: listing dipetakan ke nama HP baku dari GSMArena, lalu **jika Shopee tersedia**,
jumlah listing per HP disamakan antara Tokopedia & Shopee. Jika Shopee belum ada, perbandingan dilewati dan
seluruh listing Tokopedia yang sudah dibersihkan tetap dipakai apa adanya (tanpa penyamaan).

In [8]:
def parse_variant(title):
    for m in re.finditer(r"\b(\d{1,2})\s*(?:gb)?\s*[/+]\s*(\d{2,4})\s*(?:gb)?\b", title or "", re.I):
        ram, sto = int(m.group(1)), int(m.group(2))
        if ram <= 24 and sto in (16, 32, 64, 128, 256, 512, 1024):
            return ram, sto
    return None, None


def build_phone_keys(specs_df):
    keys = {}
    for _, r in specs_df.iterrows():
        full = normalize_name(r["phone_name"])
        no_brand = normalize_name(re.sub(rf"^{re.escape(r['brand'])}\s+", "", r["phone_name"], flags=re.I))
        klist = [full] if full else []
        if len(no_brand.split()) >= 2 and no_brand != full:
            klist.append(no_brand)
        keys[r["phone_name"]] = klist
    return keys


def match_phone(title, keys):
    t = f" {normalize_name(title)} "
    t_variants = set(t.split()) & VARIANT_TOKENS
    best, best_len = None, 0
    for phone, klist in keys.items():
        for k in klist:
            if f" {k} " in t and len(k) > best_len and t_variants <= set(k.split()):
                best, best_len = phone, len(k)
    return best


def fallback_ok(keyword, brand, title):
    base = re.sub(rf"^{re.escape(brand)}\s+", "", keyword or "", flags=re.I) if brand else (keyword or "")
    k = [w for w in normalize_name(base).split() if w not in {"5g", "4g", "3g", "lte"}]
    t = {w for w in normalize_name(title).split() if w not in {"5g", "4g", "3g", "lte"}}
    digits_k = {w for w in k if any(c.isdigit() for c in w)}
    if not digits_k or not digits_k <= t:
        return False
    return (set(k) & VARIANT_TOKENS) == (t & VARIANT_TOKENS)


def assign_phone(sales, specs, verbose=True):
    out = sales.copy()
    if out.empty:
        return out.assign(phone_name=None, listing_ram_gb=None, listing_storage_gb=None)
    keys = build_phone_keys(specs)
    brand_of = dict(zip(specs["phone_name"], specs["brand"]))
    keys_by_brand = {b: {p: keys[p] for p in specs.loc[specs["brand"] == b, "phone_name"]} for b in specs["brand"].unique()}

    def resolve(r):
        b = brand_of.get(r["search_phone"])
        m = match_phone(r["product_name"], keys_by_brand[b]) if b in keys_by_brand else None
        m = m or match_phone(r["product_name"], keys)
        if m:
            return m
        if fallback_ok(r["search_phone"], brand_of.get(r["search_phone"], ""), r["product_name"]):
            return r["search_phone"]
        return None

    out["phone_name"] = out.apply(resolve, axis=1)
    n_bad = int(out["phone_name"].isna().sum())
    if n_bad and verbose:
        print(f"⚠️ {n_bad} listing tidak cocok dengan HP di dim_phone dan dibuang (dari {len(out)}).")
    out = out.dropna(subset=["phone_name"]).copy()
    variants = out["product_name"].apply(parse_variant)
    out["listing_ram_gb"] = [v[0] for v in variants]
    out["listing_storage_gb"] = [v[1] for v in variants]
    return out.reset_index(drop=True)


def balance_platforms(sales, phone_order, k=None, min_k=None, n_phones=None, seed=42):
    k = k or LISTINGS_PER_PHONE
    min_k = min_k or MIN_LISTINGS_PER_PHONE
    n_phones = n_phones or N_PHONES_COMPARE
    cnt = sales.groupby(["phone_name", "platform"]).size().unstack(fill_value=0)
    if cnt.shape[1] < 2:
        raise RuntimeError("perbandingan butuh 2 platform")
    n_each = cnt.min(axis=1).clip(upper=k)
    n_each = n_each[n_each >= min_k]
    rank = {p: i for i, p in enumerate(phone_order)}
    chosen = sorted(n_each.index, key=lambda p: rank.get(p, 10**9))[:n_phones]
    chosen_set = set(chosen)
    parts = [g.sample(n=int(n_each[p]), random_state=seed)
             for (p, plat), g in sales.groupby(["phone_name", "platform"]) if p in chosen_set]
    out = pd.concat(parts).sort_values(["phone_name", "platform"]).reset_index(drop=True)
    print(f"HP dibandingkan: {len(chosen)} | listing per platform: {len(out) // cnt.shape[1]}")
    return out


df_sales_matched = assign_phone(df_sales_all, df_specs)
phone_order = df_specs.sort_values("release_year", ascending=False)["phone_name"].tolist()

if HAS_SHOPEE and set(df_sales_matched["platform"]) >= {"Tokopedia", "Shopee"}:
    df_sales = balance_platforms(df_sales_matched, phone_order)
else:
    print("Shopee belum tersedia/cocok — perbandingan antar platform dilewati, memakai semua listing Tokopedia apa adanya.")
    df_sales = df_sales_matched

print(f"\nTotal listing gold: {len(df_sales)}")
df_sales.head()

⚠️ 464 listing tidak cocok dengan HP di dim_phone dan dibuang (dari 1258).
HP dibandingkan: 8 | listing per platform: 45

Total listing gold: 90


,platform,search_phone,product_name,product_url,price_original,price_discount,discount_pct,is_original_estimated,rating,units_sold,seller_name,seller_location,scrape_date,phone_name,listing_ram_gb,listing_storage_gb
0,Shopee,Apple iPhone 16,iPhone 16 128gb NEW BARU SEGEL,https://shopee.co.id/iPhone-16-128gb-NEW-BARU-...,16100000,16100000,0.0,False,5.0,7,shop_207409077,Kab. Bantul,2026-10-03,Apple iPhone 16,NaN,NaN
1,Shopee,Apple iPhone 16,Apple [iPhone] 16 128GB 256GB Garansi Resmi GDN,https://shopee.co.id/Apple-iPhone-16-128GB-256...,19999000,16290000,18.5,False,4.8,52,shop_1516969716,Jakarta Barat,2026-10-03,Apple iPhone 16,NaN,NaN
2,Shopee,Apple iPhone 16,iPhone 16 128GB - Garansi Resmi Indonesia,https://shopee.co.id/iPhone-16-128GB-Garansi-R...,18990000,17490000,7.9,False,5.0,0,shop_1732890112,Jakarta Utara,2026-10-03,Apple iPhone 16,NaN,NaN
3,Shopee,Apple iPhone 16,Apple iPhone 16 128GB New Garansi Resmi Indonesia,https://shopee.co.id/Apple-iPhone-16-128GB-New...,15810345,15810345,0.0,False,5.0,6,shop_389328160,Bogor,2026-10-03,Apple iPhone 16,NaN,NaN
4,Shopee,Apple iPhone 16,Apple [iPhone] 16 128GB 256GB 512GB,https://shopee.co.id/Apple-iPhone-16-128GB-256...,19999000,16290000,18.5,False,4.9,595,shop_1516969716,Jakarta Barat,2026-10-03,Apple iPhone 16,NaN,NaN


## 7. Bangun Star Schema

In [9]:
def build_dim_date(dates):
    d = pd.to_datetime(pd.Series(sorted(set(dates))))
    return pd.DataFrame({
        "date_id": d.dt.strftime("%Y%m%d").astype(int), "full_date": d.dt.date, "day": d.dt.day,
        "month": d.dt.month, "month_name": d.dt.month_name(), "quarter": d.dt.quarter,
        "year": d.dt.year, "day_name": d.dt.day_name(),
    })


dim_brand = df_specs[["brand"]].drop_duplicates().rename(columns={"brand": "brand_name"}).reset_index(drop=True)
dim_brand.insert(0, "brand_id", dim_brand.index + 1)

dim_phone = df_specs.merge(dim_brand, left_on="brand", right_on="brand_name")
dim_phone = dim_phone[["brand_id", "phone_name", "chipset", "chipset_brand", "ram_gb", "storage_gb",
                       "battery_mah", "release_year", "gsmarena_url"]].reset_index(drop=True)
dim_phone.insert(0, "phone_id", dim_phone.index + 1)

dim_platform = df_sales[["platform"]].drop_duplicates().rename(columns={"platform": "platform_name"}).reset_index(drop=True)
dim_platform.insert(0, "platform_id", dim_platform.index + 1)

dim_seller = (df_sales[["platform", "seller_name", "seller_location"]].drop_duplicates()
              .merge(dim_platform, left_on="platform", right_on="platform_name")
              [["platform_id", "seller_name", "seller_location"]].reset_index(drop=True))
dim_seller.insert(0, "seller_id", dim_seller.index + 1)

dim_date = build_dim_date(df_sales["scrape_date"])

f = (df_sales.merge(dim_phone[["phone_id", "phone_name"]], on="phone_name", how="left")
             .merge(dim_platform, left_on="platform", right_on="platform_name", how="left")
             .merge(dim_seller, on=["platform_id", "seller_name", "seller_location"], how="left"))
f["date_id"] = pd.to_datetime(f["scrape_date"]).dt.strftime("%Y%m%d").astype(int)

fact_sales = f[["phone_id", "platform_id", "seller_id", "date_id", "product_name", "product_url",
               "listing_ram_gb", "listing_storage_gb", "price_original", "price_discount", "discount_pct",
               "is_original_estimated", "rating", "units_sold"]].reset_index(drop=True)
fact_sales.insert(0, "sales_id", fact_sales.index + 1)

for name, tbl in [("dim_brand", dim_brand), ("dim_phone", dim_phone), ("dim_platform", dim_platform),
                  ("dim_seller", dim_seller), ("dim_date", dim_date), ("fact_sales", fact_sales)]:
    print(f"{name:14s} -> {tbl.shape[0]:5d} baris, {tbl.shape[1]:2d} kolom")

dim_brand      ->    10 baris,  2 kolom
dim_phone      ->   270 baris, 10 kolom
dim_platform   ->     2 baris,  2 kolom
dim_seller     ->    54 baris,  4 kolom
dim_date       ->     3 baris,  8 kolom
fact_sales     ->    90 baris, 15 kolom


## 8. Mart: Perbandingan Harga

In [10]:
def build_price_comparison():
    fa = (fact_sales.merge(dim_phone[["phone_id", "phone_name"]], on="phone_id")
                    .merge(dim_platform, on="platform_id"))
    stats = (fa.groupby(["phone_name", "platform_name"])
              .agg(jumlah_listing=("sales_id", "count"), harga_median=("price_discount", "median"),
                   harga_rata2=("price_discount", "mean"), harga_min=("price_discount", "min"),
                   harga_maks=("price_discount", "max"), rating_rata2=("rating", "mean"),
                   total_terjual=("units_sold", "sum")).round(1).reset_index())
    if fa["platform_name"].nunique() < 2:
        return stats, pd.DataFrame()
    wide = fa.pivot_table(index="phone_name", columns="platform_name", values="price_discount", aggfunc="median")
    wide["selisih_rp"] = (wide["Shopee"] - wide["Tokopedia"]).round(0)
    wide["selisih_pct"] = (wide["selisih_rp"] / wide["Tokopedia"] * 100).round(1)
    wide["lebih_murah"] = wide["selisih_rp"].apply(lambda x: "Sama" if x == 0 else ("Shopee" if x < 0 else "Tokopedia"))
    return stats, wide.reset_index()


mart_price_stats, mart_price_comparison = build_price_comparison()
if not mart_price_comparison.empty:
    print(mart_price_comparison["lebih_murah"].value_counts().to_string())
mart_price_comparison.head(10)

lebih_murah
Tokopedia    5
Shopee       2


platform_name,phone_name,Shopee,Tokopedia,selisih_rp,selisih_pct,lebih_murah
0,Infinix Smart 20,2116723.5,2249000.0,-132276.0,-5.9,Shopee
1,Samsung Galaxy A07,2479000.0,2404350.0,74650.0,3.1,Tokopedia
2,Samsung Galaxy A17,3949000.0,5030200.0,-1081200.0,-21.5,Shopee
3,Samsung Galaxy A57,7299000.0,6849000.0,450000.0,6.6,Tokopedia
4,Xiaomi Poco C81 Pro,2108500.0,1992500.0,116000.0,5.8,Tokopedia
5,Xiaomi Redmi Note 17,3999000.0,3616486.0,382514.0,10.6,Tokopedia
6,Xiaomi Redmi Note 17 Pro Max,7208000.0,6960000.0,248000.0,3.6,Tokopedia


## 9. (Opsional) Sentimen YouTube
Dilewati otomatis jika `data/youtube.duckdb` belum ada / skemanya belum sesuai (notebook YouTube belum
disesuaikan ke arsitektur DuckDB ini).

In [ ]:
mart_sentiment_summary = pd.DataFrame()
mart_sentiment_vs_price = pd.DataFrame()

if HAS_YOUTUBE:
    try:
        fact_yt = con.execute("SELECT * FROM yt.fact_youtube_sentiment").df()
        g = fact_yt.groupby("phone_id")["sentiment"].value_counts(normalize=True).unstack(fill_value=0) * 100
        g = g.reindex(columns=["positive", "neutral", "negative"], fill_value=0).round(1)
        g.columns = [f"pct_{c}" for c in g.columns]
        g["n_comments"] = fact_yt.groupby("phone_id").size()
        mart_sentiment_summary = g.reset_index().merge(dim_phone[["phone_id", "phone_name"]], on="phone_id")
        if not mart_price_comparison.empty:
            mart_sentiment_vs_price = mart_sentiment_summary.merge(
                mart_price_comparison[["phone_name", "Tokopedia", "Shopee", "lebih_murah"]], on="phone_name", how="left")
        print(f"Sentimen YouTube ditemukan untuk {len(mart_sentiment_summary)} HP.")
    except duckdb.Error as e:
        print(f"Skema youtube.duckdb belum sesuai yang diharapkan ({e}). Bagian ini dilewati untuk sekarang.")
else:
    print("data/youtube.duckdb belum ada. Bagian ini dilewati untuk sekarang.")

## 10. Tulis ke DuckLake
`INSTALL ducklake` mengunduh ekstensi (sekali saja, butuh internet). Setelahnya, `ATTACH 'ducklake:...'` membuat
katalog lakehouse: metadatanya di `lakehouse.ducklake`, datanya sebagai file Parquet di folder `files/`.

In [11]:
con.execute("INSTALL ducklake")
con.execute("LOAD ducklake")
con.execute(f"ATTACH 'ducklake:{LAKE_CATALOG}' AS lake (DATA_PATH '{LAKE_DATA_PATH}')")

tables = {"dim_brand": dim_brand, "dim_phone": dim_phone, "dim_platform": dim_platform,
         "dim_seller": dim_seller, "dim_date": dim_date, "fact_sales": fact_sales,
         "mart_price_stats": mart_price_stats, "mart_price_comparison": mart_price_comparison}
if not mart_sentiment_summary.empty:
    tables["mart_sentiment_summary"] = mart_sentiment_summary
    tables["mart_sentiment_vs_price"] = mart_sentiment_vs_price

for name, df in tables.items():
    if df.empty:
        print(f"– {name} dilewati (kosong)")
        continue
    con.execute(f"CREATE OR REPLACE TABLE lake.{name} AS SELECT * FROM df")
    print(f"✔ lake.{name} ({len(df)} baris)")

con.execute("USE lake")

NameError: name 'mart_sentiment_summary' is not defined

## 11. Validasi & Ukuran Lakehouse

In [ ]:
checks = {
    "phone_id":    con.execute("SELECT count(*) FROM lake.fact_sales f LEFT JOIN lake.dim_phone d USING (phone_id) WHERE d.phone_id IS NULL").fetchone()[0] == 0,
    "platform_id": con.execute("SELECT count(*) FROM lake.fact_sales f LEFT JOIN lake.dim_platform d USING (platform_id) WHERE d.platform_id IS NULL").fetchone()[0] == 0,
    "seller_id":   con.execute("SELECT count(*) FROM lake.fact_sales f LEFT JOIN lake.dim_seller d USING (seller_id) WHERE d.seller_id IS NULL").fetchone()[0] == 0,
}
print("Validasi foreign key:", checks)
assert all(checks.values())

size_mb = sum(p.stat().st_size for p in LAKE_DIR.rglob("*") if p.is_file()) / 1_000_000
flag = " ⚠️ mendekati/melebihi batas GitHub!" if size_mb > GITHUB_WARN_MB else ""
print(f"Ukuran folder {LAKE_DIR}: {size_mb:.1f} MB{flag}")

con.execute("""
    SELECT p.brand_name, count(*) AS jumlah_listing, round(avg(f.price_discount)) AS avg_harga
    FROM lake.fact_sales f JOIN lake.dim_phone ph USING (phone_id) JOIN lake.dim_brand p USING (brand_id)
    GROUP BY p.brand_name ORDER BY jumlah_listing DESC
""").df()

## Catatan
- **Streamlit nanti membaca dari `data/lakehouse/lakehouse.ducklake`** (lewat `ATTACH 'ducklake:...'` yang sama),
  tidak perlu membaca file bronze sama sekali.
- **Folder `data/lakehouse/files/`** berisi file Parquet — ini yang paling mungkin membesar seiring bertambah
  data. Cek ukurannya (bagian 11) sebelum `git push`.
- **Jalankan ulang notebook ini kapan saja** setelah ada data baru di salah satu bronze (mis. setelah scraping
  Shopee dilanjutkan) — `CREATE OR REPLACE TABLE` di bagian 10 akan menimpa isi lama dengan yang terbaru.
- **Bagian sentimen YouTube** akan otomatis terisi begitu `data/youtube.duckdb` dibangun ulang ke arsitektur ini
  dan tabel `fact_youtube_sentiment`-nya tersedia — tidak perlu mengubah notebook ini lagi.